In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_agent

In [3]:
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearch(max_results=3)

In [5]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("What is the latest news on AI?")
result

{'query': 'What is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'October 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement [...] August 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulat

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
)

In [8]:
response = llm.invoke("What year is it?")
response

AIMessage(content='It’s 2026.', additional_kwargs={'reasoning_content': 'The user asks: "What year is it?" This is a simple question. The system says current date: 2026-09-07. So answer: 2026.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 76, 'total_tokens': 129, 'completion_time': 0.072037271, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.004249505, 'prompt_tokens_details': None, 'queue_time': 0.346236311, 'total_time': 0.076286776}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8e23cedc90', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07b2e-4d38-7822-9f87-7f011bd53458-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 53, 'total_tokens': 129, 'output_token_details': {'reasoning': 38}})

In [9]:
# tools = []
tools = [search_tool, get_weather]

In [10]:
# Create Agent
agent = create_agent(
    model = llm,
    tools = tools
)

In [11]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the current weather in Ongole"}
    ]
})
response

{'messages': [HumanMessage(content='Find the current weather in Ongole', additional_kwargs={}, response_metadata={}, id='8b64bd50-b0c3-4933-b603-5e731e3d3580'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get current weather in Ongole. Use get_weather function.', 'tool_calls': [{'id': 'fc_af1dcc2c-22a9-44bd-8533-8f282a6164c6', 'function': {'arguments': '{"city":"Ongole"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 343, 'total_tokens': 384, 'completion_time': 0.045419294, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.021973058, 'prompt_tokens_details': None, 'queue_time': 0.346164611, 'total_time': 0.067392352}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07b2e-5035-7440-8846-11aabac98e1e-0', tool_cal

In [12]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 38 °C  
- **Weather:** Overcast  
- **Humidity:** 33%


In [13]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in Ongole

City: Ongole
Temperature: 38°C
Weather: Overcast 
Humidity: 33%
**Current Weather in Ongole**

- **Temperature:** 38 °C  
- **Weather:** Overcast  
- **Humidity:** 33%


In [14]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about the Latest News on Iran and US War"}
    ]
})

In [15]:
for msg in response["messages"]:
    print(msg.content)

Tell me about the Latest News on Iran and US War

{"query": "latest news Iran US war", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.independent.co.uk/news/world/middle-east/iran-us-war-live-trump-hormuz-strikes-oil-tankers-b3045692.html", "title": "Iran-US war latest: Tehran vows to extend ‘exclusion zone’ near Strait of Hormuz after US strikes on tankers | The Independent", "content": "It comes as the US and Iran have traded strikes on ships, the latest round of attacks since the conflict flared a week ago after a period of relative calm.\n\nUS Central Command said it struck three Iranian oil tankers, including one off the coast of Kharg Island, near the nation’s oil export hub, after Iran's Islamic Revolutionary Guard Corps launched ballistic missiles at two US Navy ships.\n\nThe IRGC responded by threatening intensified strikes against US military vessels in the region, and later said it targeted three tankers that Tehran claimed were o